# citkid pipeline_v2 — overview and usage

This notebook walks through the simplified `citkid.pipeline_v2` framework end-to-end. The V2 pipeline keeps only one active version of every analysis product, stores the YAML and custom step source inside the zarr output, and invalidates downstream data when an earlier step is re-run.

---

## Concepts

The V2 pipeline still has the same three core layers:

| Layer | Class / function | Purpose |
|---|---|---|
| **Step definition** | `plStep` | Describes one calibration or analysis step, associated with a single function call |
| **Data + calibration** | `DataSet` | Lazy zarr-backed parameter store + calibration pipeline execution |
| **Analysis execution** | `AnalysisRunner` | Runs analysis on a `DataSet` to create analysis products |
| **Interactive review** | `run_iq_analysis`, `run_ts_analysis`, … | Interactive code for manually adjusting analysis parameters |

The main behavioral change is that there are **no runs**. If you execute steps A → B → C and then re-run B, the outputs from C are deleted. This guarantees that the dataset can never expose conflicting upstream/downstream results.

**`data_idx`** is the row index — one entry per tone.  
**`nrows`** is the total number of rows.

**`execution_mode`** controls whether vectorized steps load all data at once (`'vectorized'`, default) or loop over rows individually (`'per-row'`) for lower memory usage. See section 3a-bis for details.

---

## 1. Defining custom steps with `plStep`

Write custom calibration and analysis steps exactly as in the original pipeline. The calibration file should expose `custom_cal_steps`, and the analysis file should expose `custom_analysis_steps`.

The same `func_type` conventions apply: `'global'`, `'global-res'`, `'per-row'`, and `'vectorized'`.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline.framework import plStep

def load_global_data():
    """Load data shared across all rows."""
    root = zarr.open('path/to/data.zarr', mode='r')
    fres_all = np.array(root['fres_all'])
    qres_all = np.array(root['qres_all'])
    nrows = int(root['fres'].shape[0])
    return fres_all, qres_all, nrows

def load_global_res_data():
    """Load per-resonator data computed once for all rows."""
    root = zarr.open('path/to/data.zarr', mode='r')
    return np.array(root['fres']), np.array(root['qres']), np.array(root['ares']), np.array(root['res_idxs'])

def load_data_f(data_idx):
    """Load the fine sweep for one row."""
    root = zarr.open('path/to/data.zarr', mode='r')
    ff = np.array(root['f'][data_idx])
    zf = np.array(root['z'][data_idx])
    idx = np.argsort(ff)
    return ff[idx], zf[idx]

custom_cal_steps = [
    plStep('load_global_data', load_global_data, [], ['fres_all', 'qres_all', 'nrows'], 'global'),
    plStep('load_global_res_data', load_global_res_data, [], ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
    plStep('load_data_f', load_data_f, ['data_idx'], ['ff', 'zf'], 'per-row'),
]

def fit_example(ff, zf):
    """Example analysis step."""
    return np.nanmean(ff), np.nanmean(np.abs(zf))

custom_analysis_steps = [
    plStep('fit_example', fit_example, ['ff', 'zf'], ['fr_est', 'amp_est'], 'per-row'),
]

## 2. DataSet — lazy zarr-backed parameter store

`pipeline_v2.DataSet` is still lazy: `DS.ff` does not load anything until you index it. The difference is storage layout. All active parameters live directly at the top level of the zarr store rather than under `run1`, `run2`, and so on.

## YAML aliases

Pass these strings instead of a file path and the built-in calibration template is used:

| alias | template file |
|---|---|
| `'iq'` | `cal-iqonly.yaml` |
| `'ts'` | `cal.yaml` |
| `'ts_offres'` | `cal-offres.yaml` |

In [ ]:
from citkid.pipeline_v2.dataset import DataSet

zarr_path = 'path/to/output_v2.zarr'

DS = DataSet(
    zarr_path = zarr_path,
    cal_yaml_path = 'iq',
    custom_cal_steps = custom_cal_steps,
    # custom_path = 'my_custom_cal_steps.py',
    zarr_mode = 'a',
)

### 2a. Accessing parameters

```python
DS.nrows
DS.fres[0]
DS.ff[0]
DS.ff[[0, 1, 2]]
```

Global parameters are returned directly. Per-row parameters are exposed as lazy accessors backed by the top-level zarr arrays.

In [ ]:
print('nrows:', DS.nrows)
print('fres[0]:', DS.fres[0])

ff0 = DS.ff[0]
zf0 = DS.zf[0]

print(DS.root.tree())

### 2b. Embedded definitions

On first creation, `DataSet` stores the calibration YAML and custom calibration source inside the zarr root metadata. After that, you can re-open the dataset with only the zarr path.

In [ ]:
# Later, possibly in a new session
DS_reloaded = DataSet(zarr_path = zarr_path)
print(DS_reloaded.nrows)

## 3. AnalysisRunner — execute the analysis pipeline

`AnalysisRunner` still wraps a `DataSet` and executes a YAML-defined analysis path. The V2 change is in invalidation behavior: re-running an earlier step deletes products from later steps.

The analysis YAML and custom analysis source are also embedded into the zarr output the first time the runner is created.

In [ ]:
from citkid.pipeline_v2.analysis import AnalysisRunner

AR = AnalysisRunner(
    DS,
    analysis_yaml_path = 'iq',
    custom_path = 'my_custom_analysis_steps.py',
)

### 3a. Run the full analysis path

In [ ]:
AR.execute_path(verbose = True)

### 3a-bis. Controlling memory usage with execution_mode

For large datasets where vectorized execution would load too much data into memory, use the `execution_mode` parameter to switch from vectorized to per-row execution. This applies to both `execute_path()` and `execute_step()`.

| Mode | Behavior | Memory | Speed |
|---|---|---|---|
| `'vectorized'` (default) | Loads all row data at once for each vectorized step | High | Fast |
| `'per-row'` | Loops over each row individually, even for vectorized steps | Low | Slower |

When `execution_mode='per-row'`, vectorized steps execute one row at a time instead of collecting all data upfront. This is useful when fitting large IQ datasets that don't fit in memory.

In [ ]:
# Memory-efficient execution: loop over rows one at a time
AR.execute_path(
    execution_mode='per-row',  # Loop over each row individually
    verbose=True,
)

# Alternatively, execute a single step with per-row mode
fit_iq_step = next(s['task'] for s in AR.path if s['task'].name == 'fit_iq')
AR.execute_step(
    fit_iq_step,
    data_idx=None,
    execution_mode='per-row',
)

### 3b. Re-running a single step

If you re-run step B after step C already exists, V2 deletes C before re-executing B. This is the central simplification relative to the original pipeline.

In [ ]:
fit_iq_step = next(s['task'] for s in AR.path if s['task'].name == 'fit_iq')

custom_mask = np.ones(DS.ff[5].shape, dtype=bool)
custom_mask[:10] = False

AR.execute_step(
    fit_iq_step,
    data_idx = [5],
    user_params = {'iq_mask': custom_mask},
    save = True,
)

# Any downstream products that depended on fit_iq are now invalidated.

### 3c. Save in-memory results without re-running

Unsaved results can still accumulate in memory during interactive work. Use `save_step_outputs` to persist the current outputs for a step.

In [ ]:
AR.execute_step(fit_iq_step, data_idx = [7], user_params = {'iq_mask': None})
AR.save_step_outputs(fit_iq_step, data_idx = [7])

### 3d. Re-opening without extra file paths

Once the analysis definition is embedded, you can re-open both the dataset and runner with just the zarr path.

In [ ]:
DS2 = DataSet(zarr_path = zarr_path)
AR2 = AnalysisRunner(DS2)
print([step_dict['task'].name for step_dict in AR2.path])

## 4. Interactive analysis

The interactive UI is intentionally reused. Import the interactive entry points from `citkid.pipeline_v2.interactive` and pass in a V2 `AnalysisRunner`.

In [ ]:
from citkid.pipeline_v2.interactive import run_iq_analysis, run_ts_analysis

on_res_idxs = np.where(DS.res_idxs[:] >= 0)[0]

run_iq_analysis(
    AR,
    start_idx = 0,
    data_idxs = on_res_idxs,
    title = 'IQ Analysis V2',
    ui_scale = 1.0,
    plot_scale = 1.0,
)

## Summary

Use `pipeline_v2` when you want the same basic workflow as the original pipeline but without run management. 